# Cost analysis

Computes valid requirement-matching failures first, then estimates cost per valid failure with simple editable assumptions. This is not GPU benchmarking.


In [ ]:
from pathlib import Path
import csv, subprocess

REPO = 'https://github.com/casparbreloh/rbt4dnn-seminar.git'
DATA = Path('data/input.csv')
roots = [Path.cwd(), *Path.cwd().parents]

path = next((root / DATA for root in roots if (root / DATA).exists()), None)
if path is None:
    repo = Path('/content/rbt4dnn-seminar')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO, str(repo)], check=True)
    path = repo / DATA

if not path.exists():
    raise FileNotFoundError(f'Could not find {DATA}')

ROOT = path.parents[1]
with path.open(newline='') as f:
    rows = list(csv.DictReader(f))

print(path)


In [1]:
ASSUMPTIONS = [
    ('gpu_hour_cost_usd', 3.00, 'USD/hour'),
    ('per_requirement_train_hours', 1.00, 'hours'),
    ('shared_train_hours', 1.50, 'hours'),
    ('generation_hours_per_1000', 0.10, 'hours/1000 images'),
    ('human_validation_cost_per_image_usd', 0.03, 'USD/image'),
    ('engineer_hours_per_requirement', 2.00, 'hours'),
    ('engineer_hourly_rate_usd', 25.00, 'USD/hour'),
]
A = {k: v for k, v, _ in ASSUMPTIONS}


def f(x):
    return None if x == '' else float(x)


def write_csv(path, fieldnames, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', newline='') as out:
        writer = csv.DictWriter(out, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def estimated_cost(r):
    n = int(r['n_images'])
    if r['variant'] == 'lr':
        train = A['per_requirement_train_hours']
        eng = A['engineer_hours_per_requirement']
    else:
        train = A['shared_train_hours'] / 6
        eng = A['engineer_hours_per_requirement'] / 2
    return (
        A['gpu_hour_cost_usd'] * (train + A['generation_hours_per_1000'] * n / 1000)
        + A['human_validation_cost_per_image_usd'] * n
        + eng * A['engineer_hourly_rate_usd']
    )


out_dir = ROOT / 'data' / 'results'
write_csv(
    out_dir / 'cost-assumptions.csv',
    ['parameter', 'value', 'unit'],
    [{'parameter': k, 'value': f'{v:.2f}', 'unit': unit} for k, v, unit in ASSUMPTIONS],
)

valid_rows = []
cost_rows = []
for r in rows:
    p = f(r['pass_rate_mean'])
    m = f(r['precondition_match_mean'])
    n = int(r['n_images']) if r['n_images'] else None
    if p is None or m is None or n is None:
        continue

    valid_failures = n * m * (1 - p)
    valid_rows.append({
        'dataset': r['dataset'],
        'requirement': r['requirement'],
        'variant': r['variant'],
        'n_images': n,
        'pass_rate': f'{p:.6f}',
        'precondition_match': f'{m:.6f}',
        'valid_failures': f'{valid_failures:.3f}',
        'nonmatching_failure_share': f'{(1 - m) * (1 - p):.6f}',
    })

    cost = estimated_cost(r)
    cost_rows.append({
        'dataset': r['dataset'],
        'requirement': r['requirement'],
        'variant': r['variant'],
        'n_images': n,
        'pass_rate': f'{p:.6f}',
        'precondition_match': f'{m:.6f}',
        'valid_failures': f'{valid_failures:.3f}',
        'estimated_cost_usd': f'{cost:.2f}',
        'cost_per_valid_failure_usd': '' if valid_failures <= 0 else f'{cost / valid_failures:.2f}',
    })

valid_rows.sort(key=lambda row: float(row['valid_failures']), reverse=True)
cost_rows.sort(key=lambda row: float(row['cost_per_valid_failure_usd'] or 'inf'))

write_csv(
    out_dir / 'valid-failures.csv',
    [
        'dataset', 'requirement', 'variant', 'n_images', 'pass_rate',
        'precondition_match', 'valid_failures', 'nonmatching_failure_share',
    ],
    valid_rows,
)
write_csv(
    out_dir / 'cost-analysis.csv',
    [
        'dataset', 'requirement', 'variant', 'n_images', 'pass_rate',
        'precondition_match', 'valid_failures', 'estimated_cost_usd',
        'cost_per_valid_failure_usd',
    ],
    cost_rows,
)

print('largest estimated valid-failure yields')
for r in valid_rows[:8]:
    print(f"{r['dataset']} {r['requirement']} {r['variant']} | {float(r['valid_failures']):.1f}")

print()
print('cheapest estimated valid failures')
print('dataset req variant | valid failures | cost | cost/failure')
for r in cost_rows[:12]:
    print(
        f"{r['dataset']} {r['requirement']} {r['variant']} | "
        f"{float(r['valid_failures']):.1f} | ${float(r['estimated_cost_usd']):.2f} | "
        f"${float(r['cost_per_valid_failure_usd']):.2f}"
    )
print(out_dir / 'cost-analysis.csv')


dataset req variant | pass | match | valid failures | cost/failure
mnist M3 lr | 0.694 | 0.958 | 2930.5 | $0.12
celeba C1 lr | 0.916 | 0.961 | 809.2 | $0.44
celeba C2 lr | 0.879 | 0.964 | 1170.3 | $0.30
celeba C3 lr | 0.914 | 0.975 | 837.5 | $0.43
celeba C4 lr | 0.909 | 0.894 | 813.5 | $0.44
celeba C5 lr | 0.975 | 0.955 | 240.7 | $1.48
celeba C6 lr | 0.796 | 0.981 | 2004.2 | $0.18
sgsm S1 lr | 0.653 | 0.760 | 2633.4 | $0.14
sgsm S2 lr | 0.134 | 0.850 | 7364.4 | $0.05
sgsm S3 lr | 0.887 | 0.783 | 884.8 | $0.40
mnist M3 allreq | 0.932 | 0.275 | 18.7 | $3.00
mnist M3 alldata | 0.996 | 0.074 | 0.3 | $189.36


Rows without precondition-match data, such as M7/C7 in the paper data, are excluded from valid-failure and cost-per-valid-failure estimates. The key comparison is not raw pass rate, but useful valid failures per dollar.